In [ ]:
import torch
import torch.nn as nn

class SingleHeadAttention(nn.Module):

    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.qkv = nn.Linear(embed_dim, 3 * head_dim)
        self.head_dim = head_dim
    
    def forward(self, x, causal = True):

        B, T, E = x.shape
        # Each is (B, T, head_dim)
        q, k, v = self.qkv(x).chunk(3, dim = -1)
        # (B, T, T)
        scores = q @ k.transpose(-2, -1) * (self.head_dim ** -0.5)

        if causal:

            mask = torch.tril(torch.ones((T, T), dtype = torch.bool))
            scores = scores.masked_fill(~mask, float("-inf"))
        
        p = torch.softmax(scores, dim = -1)
        return p @ v


tensor([[ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True]])